In [1]:
import pandas as pd
from scipy import signal
import math
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.signal import butter, filtfilt

In [2]:
def import_data(file_path):
    """Import data from a *.glm file and store it in a dataframe"""

    # Colonnes réellement nécessaires
    ColNames = [
        'time (s)',

        'Fxal (N)', 'Fyal (N)', 'Fzal (N)',
        'Txal (Nm)', 'Tyal (Nm)', 'Tzal (Nm)',

        'Fxar (N)', 'Fyar (N)', 'Fzar (N)',
        'Txar (Nm)', 'Tyar (Nm)', 'Tzar (Nm)',

        'GF (N)', 'LFv (N)', 'LFh (N)', 'LFt (N)'
    ]

    # Noms simplifiés
    NewColNames = [
        'time',

        'Fxal','Fyal','Fzal',
        'Txal','Tyal','Tzal',

        'Fxar','Fyar','Fzar',
        'Txar','Tyar','Tzar',

        'GF','LFv','LFh','LFt'
    ]

    df = pd.read_csv(
        file_path,
        sep="\t",
        header=0,
        usecols=ColNames,
        engine="python"
    )

    df.columns = NewColNames

    return df


def compute_cop(glm_df, baseline, z0=0.00155, angle=0.523599):
    """Compute center of pressure (CPL, CPR)"""

    # Forces ATI frame
    Fal = -np.array([
        glm_df['Fxal'],
        glm_df['Fyal'],
        glm_df['Fzal']
    ])

    Far = -np.array([
        glm_df['Fxar'],
        glm_df['Fyar'],
        glm_df['Fzar']
    ])

    Tal = -np.array([
        glm_df['Txal'],
        glm_df['Tyal'],
        glm_df['Tzal']
    ])

    Tar = -np.array([
        glm_df['Txar'],
        glm_df['Tyar'],
        glm_df['Tzar']
    ])

    # Baseline correction
    Fal = Fal - np.nanmean(Fal[:, baseline], axis=1).reshape((3,1))
    Far = Far - np.nanmean(Far[:, baseline], axis=1).reshape((3,1))

    Tal = Tal - np.nanmean(Tal[:, baseline], axis=1).reshape((3,1))
    Tar = Tar - np.nanmean(Tar[:, baseline], axis=1).reshape((3,1))

    # CoP calculation
    CPL_ati = -np.array([
        (Tal[1,:] + Fal[0,:]*z0) / Fal[2,:],
        -(Tal[0,:] - Fal[1,:]*z0) / Fal[2,:]
    ])

    CPR_ati = -np.array([
        (Tar[1,:] + Far[0,:]*z0) / Far[2,:],
        -(Tar[0,:] - Far[1,:]*z0) / Far[2,:]
    ])

    # Conversion repère manipulandum
    CPL = -CPL_ati[0,:]*math.sin(angle) - CPL_ati[1,:]*math.cos(angle)
    CPR =  CPR_ati[0,:]*math.sin(angle) + CPR_ati[1,:]*math.cos(angle)

    return CPL, CPR

In [4]:
# =========================
# PARAMÈTRES
# =========================
dossier = "Groupe4_LUNDI_APREM"

fichiers_S01 = [
    "Gr4_S01_L1_001.glm", "Gr4_S01_L1_002.glm", "Gr4_S01_L1_003.glm",
    "Gr4_S01_L1_004.glm", 
    "Gr4_S01_L2_001.glm", "Gr4_S01_L2_002.glm", "Gr4_S01_L2_003.glm",
    "Gr4_S01_L2_004.glm",
    "Gr4_S01_L3_001.glm", "Gr4_S01_L3_002.glm", "Gr4_S01_L3_003.glm",
    "Gr4_S01_L3_004.glm",
    "Gr4_S01_L4_001.glm", "Gr4_S01_L4_002.glm", "Gr4_S01_L4_003.glm",
    "Gr4_S01_L4_004.glm",
]

fs = 800
cutoff = 40

# ⚠️ À ADAPTER SELON TES DONNÉES
tmin = 0
tmax = 10

# =========================
# FILTRE PASSE-BAS
# =========================
def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq

    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data)

processed_data = []

for file_name in fichiers_S01:

    file_path = os.path.join(dossier, file_name)

    if not os.path.exists(file_path):
        print(f"⚠️ Fichier non trouvé : {file_name}")
        continue

    df = import_data(file_path)

    time = df["time"].values

    # baseline = premières 0.5 secondes
    baseline = np.where(time < 0.5)[0]

    # =========================
    # COP
    # =========================
    CPL, CPR = compute_cop(df, baseline)

    # =========================
    # FORCES
    # =========================
    F_left = np.sqrt(
        df["Fxal"].values**2 +
        df["Fyal"].values**2 +
        df["Fzal"].values**2
    )

    F_right = np.sqrt(
        df["Fxar"].values**2 +
        df["Fyar"].values**2 +
        df["Fzar"].values**2
    )

    F_mean = (F_left + F_right) / 2
    GF = df["GF"].values

    # =========================
    # FILTRAGE
    # =========================
    F_mean = butter_lowpass_filter(F_mean, cutoff, fs)
    GF = butter_lowpass_filter(GF, cutoff, fs)
    CPL = butter_lowpass_filter(CPL, cutoff, fs)
    CPR = butter_lowpass_filter(CPR, cutoff, fs)

    # =========================
    # FENÊTRE TEMPORELLE
    # =========================
    mask = (time >= tmin) & (time <= tmax)

    processed_data.append({
        "file": file_name,
        "time": time[mask],
        "F": F_mean[mask],
        "GF": GF[mask],
        "CPL": CPL[mask],
        "CPR": CPR[mask]
    })

    print(f"✅ {file_name} processed")

print("🎉 Toutes les données sont traitées")

✅ Gr4_S01_L1_001.glm processed
✅ Gr4_S01_L1_002.glm processed
✅ Gr4_S01_L1_003.glm processed
✅ Gr4_S01_L1_004.glm processed
✅ Gr4_S01_L2_001.glm processed
✅ Gr4_S01_L2_002.glm processed
✅ Gr4_S01_L2_003.glm processed
✅ Gr4_S01_L2_004.glm processed
✅ Gr4_S01_L3_001.glm processed
✅ Gr4_S01_L3_002.glm processed
✅ Gr4_S01_L3_003.glm processed
✅ Gr4_S01_L3_004.glm processed
✅ Gr4_S01_L4_001.glm processed
✅ Gr4_S01_L4_002.glm processed
✅ Gr4_S01_L4_003.glm processed
✅ Gr4_S01_L4_004.glm processed
🎉 Toutes les données sont traitées


In [34]:
tmin = 15
tmax = 20

output_folder = "grip_S01"
for data in processed_data:

    file_name = data["file"]

    time_plot = data["time"]
    F_plot = data["F"]
    GF_plot = data["GF"]
    CPL_plot = data["CPL"]
    CPR_plot = data["CPR"]

    plt.figure(figsize=(12,8))

    # Force totale
    plt.subplot(3,1,1)
    plt.plot(time_plot, F_plot, label="Mean |F|")
    plt.title(f"Force magnitude ({tmin}-{tmax} s)")
    plt.xlabel("Time (s)")
    plt.ylabel("Force (N)")
    plt.legend()

    # COP
    plt.subplot(3,1,2)
    plt.plot(time_plot, CPL_plot, label="CPL")
    plt.plot(time_plot, CPR_plot, label="CPR")
    plt.title("Center of Pressure")
    plt.xlabel("Time (s)")
    plt.ylabel("Position (m)")
    plt.legend()

    # Grip force
    plt.subplot(3,1,3)
    plt.plot(time_plot, GF_plot)
    plt.title("Grip Force")
    plt.xlabel("Time (s)")
    plt.ylabel("Force (N)")

    plt.tight_layout()

    parts = file_name.split('_')
    groupe = parts[1]
    ligne = parts[2]
    numero = str(int(parts[3].split('.')[0]))

    simple_name = f"{groupe}_{ligne}_{numero}.png"
    save_path = os.path.join(output_folder, simple_name)

    plt.savefig(save_path)
    plt.close()

    print(f"✅ Sauvegardé : {simple_name}")

✅ Sauvegardé : S01_L1_1.png
✅ Sauvegardé : S01_L1_2.png
✅ Sauvegardé : S01_L1_3.png
✅ Sauvegardé : S01_L1_4.png
✅ Sauvegardé : S01_L2_1.png
✅ Sauvegardé : S01_L2_2.png
✅ Sauvegardé : S01_L2_3.png
✅ Sauvegardé : S01_L2_4.png
✅ Sauvegardé : S01_L3_1.png
✅ Sauvegardé : S01_L3_2.png
✅ Sauvegardé : S01_L3_3.png
✅ Sauvegardé : S01_L3_4.png
✅ Sauvegardé : S01_L4_1.png
✅ Sauvegardé : S01_L4_2.png
✅ Sauvegardé : S01_L4_3.png
✅ Sauvegardé : S01_L4_4.png


In [31]:
# dossier où sauvegarder les figures moyennes
output_folder_mean = "grip_S01"
os.makedirs(output_folder_mean, exist_ok=True)

# dictionnaire pour stocker les données par condition
data_par_L = {}

for data in processed_data:

    file_name = data["file"]
    ligne = file_name.split('_')[2]

    if ligne not in data_par_L:
        data_par_L[ligne] = {
            "F": [],
            "CPL": [],
            "CPR": [],
            "GF": [],
            "time": []
        }

    data_par_L[ligne]["F"].append(data["F"])
    data_par_L[ligne]["CPL"].append(data["CPL"])
    data_par_L[ligne]["CPR"].append(data["CPR"])
    data_par_L[ligne]["GF"].append(data["GF"])
    data_par_L[ligne]["time"].append(data["time"])


# -----------------------------
# calcul des moyennes
# -----------------------------

for ligne, d in data_par_L.items():

    min_len = min(len(arr) for arr in d["F"])

    F_stack   = np.vstack([arr[:min_len] for arr in d["F"]])
    CPL_stack = np.vstack([arr[:min_len] for arr in d["CPL"]])
    CPR_stack = np.vstack([arr[:min_len] for arr in d["CPR"]])
    GF_stack  = np.vstack([arr[:min_len] for arr in d["GF"]])
    time = d["time"][0][:min_len]

    F_mean   = np.mean(F_stack, axis=0)
    CPL_mean = np.mean(CPL_stack, axis=0)
    CPR_mean = np.mean(CPR_stack, axis=0)
    GF_mean  = np.mean(GF_stack, axis=0)

    # -----------------------------
    # FIGURE
    # -----------------------------

    plt.figure(figsize=(12,8))

    plt.subplot(3,1,1)
    plt.plot(time, F_mean, label="Mean |F|")
    plt.title(f"Mean Force magnitude {ligne}")
    plt.xlabel("Time (s)")
    plt.ylabel("Force (N)")
    plt.legend()

    plt.subplot(3,1,2)
    plt.plot(time, CPL_mean, label="CPL")
    plt.plot(time, CPR_mean, label="CPR")
    plt.title(f"Center of Pressure moyennes {ligne}")
    plt.xlabel("Time (s)")
    plt.ylabel("Position (m)")
    plt.legend()

    plt.subplot(3,1,3)
    plt.plot(time, GF_mean)
    plt.title(f"Grip Force moyenne {ligne}")
    plt.xlabel("Time (s)")
    plt.ylabel("Force (N)")

    plt.tight_layout()

    save_name = f"{ligne}_moyenne.png"
    save_path = os.path.join(output_folder_mean, save_name)

    plt.savefig(save_path)
    plt.close()

    print(f"✅ Figure moyenne sauvegardée pour {ligne} → {save_name}")

✅ Figure moyenne sauvegardée pour L1 → L1_moyenne.png
✅ Figure moyenne sauvegardée pour L2 → L2_moyenne.png
✅ Figure moyenne sauvegardée pour L3 → L3_moyenne.png
✅ Figure moyenne sauvegardée pour L4 → L4_moyenne.png


In [30]:
# dossier de sortie
output_folder = "phase_diagram_S01"
os.makedirs(output_folder, exist_ok=True)

# fenêtre temporelle
tmin = 15
tmax = 20

for data in processed_data:

    file_name = data["file"]

    time = data["time"]
    F = data["F"]
    GF = data["GF"]

    mask = (time >= tmin) & (time <= tmax)

    F_plot = F[mask]
    GF_plot = GF[mask]

    # -----------------------------
    # FIGURE
    # -----------------------------

    plt.figure(figsize=(8,6))

    plt.plot(F_plot, GF_plot, label="GF vs mean |F|")

    plt.xlabel("Force magnitude (N)")
    plt.ylabel("Grip Force (N)")
    plt.title(f"Phase Diagram ({tmin}-{tmax} s)")

    plt.legend()
    plt.grid()

    parts = file_name.split('_')
    groupe = parts[1]
    ligne = parts[2]
    numero = str(int(parts[3].split('.')[0]))

    simple_name = f"{groupe}_{ligne}_{numero}_phase.png"
    save_path = os.path.join(output_folder, simple_name)

    plt.savefig(save_path)
    plt.close()

    print(f"✅ Sauvegardé : {simple_name}")

✅ Sauvegardé : S01_L1_1_phase.png
✅ Sauvegardé : S01_L1_2_phase.png
✅ Sauvegardé : S01_L1_3_phase.png
✅ Sauvegardé : S01_L1_4_phase.png
✅ Sauvegardé : S01_L2_1_phase.png
✅ Sauvegardé : S01_L2_2_phase.png
✅ Sauvegardé : S01_L2_3_phase.png
✅ Sauvegardé : S01_L2_4_phase.png
✅ Sauvegardé : S01_L3_1_phase.png
✅ Sauvegardé : S01_L3_2_phase.png
✅ Sauvegardé : S01_L3_3_phase.png
✅ Sauvegardé : S01_L3_4_phase.png
✅ Sauvegardé : S01_L4_1_phase.png
✅ Sauvegardé : S01_L4_2_phase.png
✅ Sauvegardé : S01_L4_3_phase.png
✅ Sauvegardé : S01_L4_4_phase.png
